# 2. Embedding Filtration

**Purpose:** Single source of truth for all pre-analysis filtering of embedding data

This notebook takes raw embeddings from `1. embedding_artifacts_collection.ipynb` and produces filtered datasets ready for downstream analysis.

## Filter Phases

| Phase | Filter Name | Description |
|-------|-------------|-------------|
| **1a** | Empty Content Filter | Remove artifacts with zero/missing embeddings |
| **1b** | Single-Artifact Repo Filter | Remove ALL repos with only 1 artifact |
| **1c** | README Artifact Filter | Remove artifacts matching `readme*` pattern |
| **1d** | Repository Size Outlier | Analyze and optionally filter large repos |
| **1e** | LOF Embedding Outlier | Remove embedding outliers with sensitivity analysis |

---
## 1. Setup & Configuration

In [ ]:
# =============================================================================
# IMPORTS
# =============================================================================
import numpy as np
import pandas as pd
import json
import re
import pickle
from pathlib import Path
from datetime import datetime

# Machine Learning
from sklearn.neighbors import LocalOutlierFactor

# Visualization
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [ ]:
# =============================================================================
# FILTER CONFIGURATION
# =============================================================================
# Edit these parameters to customize the filtering pipeline

# -----------------------------------------------------------------------------
# Phase 1a: Empty Content Filter (always enabled)
# -----------------------------------------------------------------------------
# No configuration needed - automatically removes artifacts with missing embeddings

# -----------------------------------------------------------------------------
# Phase 1b: Single-Artifact Repository Filter
# -----------------------------------------------------------------------------
# Repositories with only 1 artifact are unlikely to represent meaningful
# AI tool usage patterns - they're typically auto-generated or placeholder repos.
# Remove ALL single-artifact repositories regardless of artifact name.
FILTER_SINGLE_ARTIFACT_REPOS = True

# -----------------------------------------------------------------------------
# Phase 1c: README Artifact Filter (pattern-based)
# -----------------------------------------------------------------------------
FILTER_READMES = True

# Case-insensitive regex pattern to match README files
FILTER_FILE_PATTERN = r'^(readme|license|changelog|pull_request_template|merge_request_template|issue_template|contributing|code_of_conduct|security|compass-webapp).*'

# -----------------------------------------------------------------------------
# Phase 1d: Repository Size Outlier Handling
# -----------------------------------------------------------------------------
# Method options:
#   'none'       - No filtering, just visualize the distribution
#   'iqr'        - Remove repos with artifacts > Q3 + multiplier * IQR
#   'percentile' - Remove repos above the specified percentile
#   'manual'     - Remove repos with more than N artifacts

REPO_OUTLIER_METHOD = 'percentile'  # Default: visualize only, no filtering

# Parameters for each method:
REPO_OUTLIER_IQR_MULTIPLIER = 1.5    # For 'iqr' method
REPO_OUTLIER_PERCENTILE = 95         # For 'percentile' method
REPO_OUTLIER_MAX_ARTIFACTS = 300     # For 'manual' method

# -----------------------------------------------------------------------------
# Phase 1e: LOF Embedding Outlier Filter
# -----------------------------------------------------------------------------
REMOVE_LOF_OUTLIERS = True

# Contamination: proportion of data to remove (0.0 to 0.5)
LOF_CONTAMINATION = 'auto'  # Remove top 5% outliers

# n_neighbors: Auto-calculated based on dataset size
# Formula: min(50, max(10, int(sqrt(n_samples) / 2)))
# This balances local sensitivity with stability
LOF_N_NEIGHBORS_AUTO = True  # If True, calculate automatically

# Manual override (only used if LOF_N_NEIGHBORS_AUTO = False)
LOF_N_NEIGHBORS_MANUAL = 20

# Whitelist protection: prevent known AI artifacts from being removed
PROTECT_KNOWN_AI_ARTIFACTS = True  # Uses patterns from Artifacts/*.json

In [ ]:
# =============================================================================
# PROTECTED ARTIFACTS - AI Tool Patterns (Auto-loaded from Artifacts/*.json)
# =============================================================================
# These artifacts will NEVER be filtered out, regardless of LOF scores or other conditions.
# Patterns are loaded from the tool configuration files in Artifacts/ folder.

import json
import re
from pathlib import Path

def load_protected_patterns():
    """Load AI tool artifact patterns from Artifacts/*.json files."""
    artifacts_dir = Path("../Artifacts")
    protected_exact = set()      # Exact filename matches (case-insensitive)
    protected_extensions = set() # File extensions to protect
    protected_prefixes = set()   # Path prefixes to protect

    for json_file in artifacts_dir.glob("*.json"):
        with open(json_file, 'r') as f:
            config = json.load(f)

        tool_name = config.get('tool_name', json_file.stem)

        # Add root_files (e.g., CLAUDE.md, .cursorrules, AGENTS.md)
        for root_file in config.get('root_files', []):
            protected_exact.add(root_file.lower())

        # Add config folders as protected prefixes
        for folder in config.get('config_folders', []):
            protected_prefixes.add(folder.lower().rstrip('/'))

        # Add patterns from artifact_patterns
        for pattern in config.get('artifact_patterns', []):
            # Exact paths
            if 'exact_path' in pattern:
                exact = pattern['exact_path']
                protected_exact.add(Path(exact).name.lower())

            # File extensions from file_type
            if pattern.get('file_type') == 'mdc':
                protected_extensions.add('.mdc')

            # Protect files in config folders
            if 'path_prefix' in pattern:
                protected_prefixes.add(pattern['path_prefix'].lower().rstrip('/'))

    return protected_exact, protected_extensions, protected_prefixes

# Load patterns
PROTECTED_EXACT, PROTECTED_EXTENSIONS, PROTECTED_PREFIXES = load_protected_patterns()

print("=== Protected AI Artifact Patterns ===")
print(f"\nExact filenames ({len(PROTECTED_EXACT)}):")
for name in sorted(PROTECTED_EXACT):
    print(f"  - {name}")

print(f"\nProtected extensions ({len(PROTECTED_EXTENSIONS)}):")
for ext in sorted(PROTECTED_EXTENSIONS):
    print(f"  - {ext}")

print(f"\nProtected path prefixes ({len(PROTECTED_PREFIXES)}):")
for prefix in sorted(PROTECTED_PREFIXES):
    print(f"  - {prefix}/")

def is_protected_artifact(artifact_name, artifact_path=None):
    """Check if an artifact should be protected from filtering."""
    name_lower = artifact_name.lower()

    # Check exact filename match
    if name_lower in PROTECTED_EXACT:
        return True

    # Check for "agent" keyword in artifact name (catches AGENT.md, agent_*.md, etc.)
    if 'agent' in name_lower:
        return True

    # Check extension
    for ext in PROTECTED_EXTENSIONS:
        if name_lower.endswith(ext):
            return True

    # Check path prefix (if path available)
    if artifact_path:
        path_lower = artifact_path.lower()
        for prefix in PROTECTED_PREFIXES:
            if prefix in path_lower:
                return True

    return False

In [ ]:
# =============================================================================
# HELPER FUNCTIONS
# =============================================================================

def calculate_auto_n_neighbors(n_samples: int) -> int:
    """
    Calculate optimal n_neighbors for LOF based on dataset size.

    Rationale:
    - Too small (5-10): Sensitive to noise, unstable
    - Too large (100+): May miss local outliers, computationally expensive
    - sqrt(n)/2: Balances local sensitivity with stability

    Bounds:
    - Minimum 10: Ensures enough neighbors for density estimation
    - Maximum 50: Prevents over-smoothing and keeps computation reasonable
    """
    return min(50, max(10, int(np.sqrt(n_samples) / 2)))


def generate_sensitivity_test_values(auto_n_neighbors: int) -> list[int]:
    """
    Generate test values for LOF sensitivity analysis.

    Creates 4-5 evenly spaced values from 10 to auto_n_neighbors,
    always including both endpoints.
    """
    if auto_n_neighbors <= 15:
        # Small range: just use 3 values
        step = (auto_n_neighbors - 10) // 2
        if step < 1:
            return [10, auto_n_neighbors]
        return [10, 10 + step, auto_n_neighbors]
    else:
        # Larger range: use 4-5 evenly spaced values
        n_values = 5 if auto_n_neighbors >= 40 else 4
        step = (auto_n_neighbors - 10) // (n_values - 1)
        values = [10 + i * step for i in range(n_values - 1)]
        values.append(auto_n_neighbors)  # Always include the final value
        return values


def create_before_after_chart(before: int, after: int, title: str, 
                               before_label: str = "Before", 
                               after_label: str = "After") -> go.Figure:
    """Create a simple before/after bar chart."""
    removed = before - after
    pct_removed = (removed / before * 100) if before > 0 else 0
    
    fig = go.Figure(data=[
        go.Bar(
            x=[before_label, after_label],
            y=[before, after],
            text=[f"{before:,}", f"{after:,}<br>(-{removed:,}, {pct_removed:.1f}%)"],
            textposition='outside',
            marker_color=['#636EFA', '#00CC96']
        )
    ])
    
    fig.update_layout(
        title=title,
        yaxis_title="Count",
        showlegend=False,
        height=400
    )
    
    return fig


def load_ai_artifact_patterns(artifacts_dir: Path) -> dict:
    """
    Load all AI artifact patterns from JSON config files.
    
    Returns:
        Dictionary with 'exact_names' (set) and 'patterns' (list of regex)
    """
    exact_names = set()
    patterns = []
    
    json_files = list(artifacts_dir.glob('*.json'))
    
    for json_file in json_files:
        try:
            with open(json_file) as f:
                config = json.load(f)
            
            # Extract from artifact_patterns array
            if 'artifact_patterns' in config:
                for artifact in config['artifact_patterns']:
                    # Add exact paths
                    if 'exact_path' in artifact:
                        exact_names.add(artifact['exact_path'].lower())
                    
                    # Add pattern (file name)
                    if 'pattern' in artifact:
                        pattern_str = artifact['pattern'].lower()
                        # If it's a simple file name (no wildcards), add as exact
                        if '*' not in pattern_str:
                            exact_names.add(pattern_str)
                            # Also add just the file name without path
                            if '/' in pattern_str:
                                exact_names.add(pattern_str.split('/')[-1])
                        else:
                            # Convert glob pattern to regex
                            regex_pattern = pattern_str.replace('.', r'\.').replace('*', '.*').replace('**/', '(.*/)?')
                            patterns.append(re.compile(regex_pattern, re.IGNORECASE))
                    
                    # Add glob patterns
                    if 'glob_pattern' in artifact:
                        glob_str = artifact['glob_pattern'].lower()
                        regex_pattern = glob_str.replace('.', r'\.').replace('**/', '(.*/)?').replace('*', '[^/]*')
                        patterns.append(re.compile(regex_pattern, re.IGNORECASE))
            
            # Also check root_files and config_folders
            if 'root_files' in config:
                for rf in config['root_files']:
                    exact_names.add(rf.lower())
            
            if 'config_folders' in config:
                for cf in config['config_folders']:
                    # Add as pattern to match anything in this folder
                    folder_pattern = cf.lower().rstrip('/') + '/.*'
                    patterns.append(re.compile(folder_pattern, re.IGNORECASE))
                    
        except (json.JSONDecodeError, KeyError) as e:
            print(f"  Warning: Error parsing {json_file.name}: {e}")
            continue
    
    return {'exact_names': exact_names, 'patterns': patterns}


def is_known_ai_artifact(artifact_name: str, artifact_path: str, 
                          ai_patterns: dict) -> bool:
    """Check if an artifact matches known AI artifact patterns."""
    name_lower = artifact_name.lower()
    path_lower = artifact_path.lower() if artifact_path else ''
    
    # Check exact matches
    if name_lower in ai_patterns['exact_names']:
        return True
    if path_lower in ai_patterns['exact_names']:
        return True
    
    # Check patterns against both name and path
    for pattern in ai_patterns['patterns']:
        if pattern.search(name_lower) or (path_lower and pattern.search(path_lower)):
            return True
    
    return False

In [ ]:
# =============================================================================
# PATHS CONFIGURATION
# =============================================================================
PROJECT_ROOT = Path.cwd().parent
OUTPUT_DIR = (PROJECT_ROOT.parent / 'collector' / 'output').resolve()
DATA_DIR = PROJECT_ROOT / 'data'
ARTIFACTS_DIR = PROJECT_ROOT / 'Artifacts'

# Output files
OUTPUT_COMMON_EMBEDDINGS = DATA_DIR / 'filtered_common_embeddings.npz'
OUTPUT_COMMON_METADATA = DATA_DIR / 'filtered_common_metadata.csv'
OUTPUT_AI_TOOLS_EMBEDDINGS = DATA_DIR / 'filtered_ai_tools_embeddings.npz'
OUTPUT_AI_TOOLS_METADATA = DATA_DIR / 'filtered_ai_tools_metadata.csv'
OUTPUT_FILTER_REPORT = DATA_DIR / 'filter_report.json'

print(f"Project root: {PROJECT_ROOT}")
print(f"Output directory (input): {OUTPUT_DIR}")
print(f"Data directory (output): {DATA_DIR}")
print(f"Artifacts directory: {ARTIFACTS_DIR}")

---
## 2. Load Data

In [ ]:
# =============================================================================
# DATA LOADING FUNCTIONS
# =============================================================================

def discover_repositories(output_dir: Path) -> list[tuple[str, str]]:
    """Find all org/repo pairs that have embeddings files."""
    repos = []
    for org in output_dir.iterdir():
        if not org.is_dir() or org.name.startswith('.'):
            continue
        for repo in org.iterdir():
            if not repo.is_dir() or repo.name == 'Artifacts':
                continue
            if list(repo.glob('*_embeddings.pkl')):
                repos.append((org.name, repo.name))
    return sorted(repos)


def load_repository_data(output_dir: Path, org_name: str, repo_name: str) -> dict:
    """Load embeddings, metadata, and repo metrics for a single repository."""
    repo_dir = output_dir / org_name / repo_name

    # Embeddings
    emb_file = next(repo_dir.glob('*_embeddings.pkl'), None)
    if not emb_file:
        raise FileNotFoundError(f"No embeddings file in {repo_dir}")
    with open(emb_file, 'rb') as f:
        emb_data = pickle.load(f)

    # File artifacts metadata
    meta_file = next(repo_dir.glob('*_file_artifacts.csv'), None)
    if not meta_file:
        raise FileNotFoundError(f"No metadata file in {repo_dir}")
    metadata_df = pd.read_csv(meta_file)

    # Repo metrics
    metrics = {}
    metrics_file = next(repo_dir.glob('*_repo_metrics.json'), None)
    if metrics_file:
        with open(metrics_file) as f:
            metrics = json.load(f)

    return {
        'org_name': org_name,
        'repo_name': repo_name,
        'full_name': f"{org_name}/{repo_name}",
        'file_ids': emb_data['file_ids'],
        'embeddings': emb_data['embeddings'],
        'model': emb_data.get('model', 'unknown'),
        'metadata': metadata_df,
        'repo_metrics': metrics,
    }


# Discover repositories
print("Discovering repositories...")
repositories = discover_repositories(OUTPUT_DIR)
print(f"Found {len(repositories)} repositories with embeddings")

In [ ]:
# Load all repositories
all_repo_data = []
for org, repo in repositories:
    try:
        data = load_repository_data(OUTPUT_DIR, org, repo)
        all_repo_data.append(data)
        print(f"Loaded {data['full_name']}: {len(data['file_ids'])} files")
    except Exception as e:
        print(f"Error loading {org}/{repo}: {e}")

print(f"\nSuccessfully loaded {len(all_repo_data)} repositories")

In [ ]:
# Combine all data - uses embeddings.shape[0] as source of truth
all_file_ids = []
all_embeddings = []
all_metadata = []
all_repo_labels = []

for data in all_repo_data:
    full_name = data['full_name']
    meta = data['metadata'].copy()
    n_embeddings = data['embeddings'].shape[0]  # Use embeddings count as source of truth
    
    # Get file IDs - prefer from embeddings pickle, ensure same length
    if len(data['file_ids']) == n_embeddings:
        file_ids_to_use = data['file_ids']
    elif 'file_id' in meta.columns and len(meta) == n_embeddings:
        file_ids_to_use = meta['file_id'].tolist()
    else:
        # Generate sequential IDs based on embeddings count
        file_ids_to_use = [f"file_{i:03d}" for i in range(n_embeddings)]
        print(f"\u26a0\ufe0f {full_name}: Generated {len(file_ids_to_use)} file IDs to match embeddings")
    
    # Align metadata to embeddings count
    if len(meta) != n_embeddings:
        print(f"\u26a0\ufe0f {full_name}: Adjusting metadata from {len(meta)} to {n_embeddings} rows")
        if len(meta) > n_embeddings:
            meta = meta.head(n_embeddings)
        else:
            # Pad metadata if needed (shouldn't happen normally)
            while len(meta) < n_embeddings:
                meta = pd.concat([meta, meta.iloc[[-1]]], ignore_index=True)
    
    # Create unique file IDs with repo prefix
    for fid in file_ids_to_use:
        all_file_ids.append(f"{full_name}/{fid}")
        all_repo_labels.append(full_name)
    
    all_embeddings.append(data['embeddings'])
    
    # Add identifying columns to metadata
    meta['org_name'] = data['org_name']
    meta['repo_name'] = full_name
    meta['global_file_id'] = [f"{full_name}/{fid}" for fid in file_ids_to_use]
    
    # Add repo metrics as columns
    metrics = data['repo_metrics']
    meta['total_commits'] = metrics.get('total_commits', 0)
    meta['total_authors'] = metrics.get('total_authors', 0)
    meta['commits_last_year'] = metrics.get('commits_last_year', 0)
    meta['last_commit_date'] = metrics.get('last_commit_date', '')
    
    all_metadata.append(meta)

# Stack embeddings
embeddings = np.vstack(all_embeddings)

# Combine metadata
metadata = pd.concat(all_metadata, ignore_index=True)

# Ensure we have required columns
if 'artifact_name' not in metadata.columns and 'file_name' in metadata.columns:
    metadata['artifact_name'] = metadata['file_name']
    
if 'artifact_path' not in metadata.columns and 'file_path' in metadata.columns:
    metadata['artifact_path'] = metadata['file_path']

# Store original metadata for later comparison
original_metadata = metadata.copy()

print(f"\nCombined data:")
print(f"  - Total files: {len(all_file_ids)}")
print(f"  - Embeddings shape: {embeddings.shape}")
print(f"  - Metadata rows: {len(metadata)}")
print(f"  - Unique orgs: {metadata['org_name'].nunique()}")
print(f"  - Unique repos: {metadata['repo_name'].nunique()}")
print(f"  - Unique artifact names: {metadata['artifact_name'].nunique()}")

# Validate alignment
assert len(embeddings) == len(metadata), \
    f"Mismatch: {len(embeddings)} embeddings vs {len(metadata)} metadata rows"
print("\u2713 Embeddings and metadata are aligned")

# Store initial counts for report
filter_report = {
    'timestamp': datetime.now().isoformat(),
    'input': {
        'total_artifacts': len(metadata),
        'total_repos': metadata['repo_name'].nunique(),
        'total_orgs': metadata['org_name'].nunique()
    },
    'phases': {}
}

---
## Presentation Charts: Repository Artifacts Overview (Pre-Filtering)

Generate descriptive statistics charts for the research presentation before any filtering is applied.

In [ ]:
# =============================================================================
# CHART 1: Repositories by Number of Artifacts (Pre-Filtering)
# =============================================================================
print("=== Chart 1: Repository Size Distribution (Before Filtering) ===")

# Calculate artifacts per repository using original_metadata (loaded before filtering)
repo_artifact_counts = original_metadata.groupby('repo_name').size().reset_index(name='artifact_count')

print(f"Total repositories: {len(repo_artifact_counts):,}")
print(f"Total artifacts: {repo_artifact_counts['artifact_count'].sum():,}")
print(f"\nStatistics:")
print(f"  Mean: {repo_artifact_counts['artifact_count'].mean():.1f} artifacts/repo")
print(f"  Median: {repo_artifact_counts['artifact_count'].median():.1f} artifacts/repo")
print(f"  Min: {repo_artifact_counts['artifact_count'].min()}")
print(f"  Max: {repo_artifact_counts['artifact_count'].max()}")

# Create bins for the histogram
# bins edges: [0, 1, 2, 5, 10, 20, 50, 100, 500, 2000] = 10 edges, need 9 labels
bin_edges = [0, 1, 2, 5, 10, 20, 50, 100, 500, 2000]
bin_labels = ['1', '2', '3-5', '6-10', '11-20', '21-50', '51-100', '101-500', '500+']

repo_artifact_counts['size_bin'] = pd.cut(
    repo_artifact_counts['artifact_count'], 
    bins=bin_edges,
    labels=bin_labels
)

bin_counts = repo_artifact_counts['size_bin'].value_counts().sort_index()

# Create bar chart
fig1 = go.Figure(go.Bar(
    x=bin_counts.index.astype(str),
    y=bin_counts.values,
    text=bin_counts.values,
    textposition='outside',
    marker_color='#636EFA'
))

fig1.update_layout(
    title="Distribution of Repositories by Number of Artifacts",
    xaxis_title="Number of Artifacts per Repository",
    yaxis_title="Number of Repositories",
    height=450,
    showlegend=False,
    margin=dict(t=50),  # Add top margin for labels
    yaxis=dict(range=[0, bin_counts.max() * 1.15])  # Extend y-axis for text labels
)

fig1.show()

# Print percentage breakdown
print("\nBreakdown by size:")
for bin_label, count in bin_counts.items():
    pct = count / len(repo_artifact_counts) * 100
    print(f"  {bin_label} artifacts: {count:,} repos ({pct:.1f}%)")

In [ ]:
# =============================================================================
# CHART 2: Repositories by AI Tools (Pre-Filtering)
# =============================================================================
from collections import defaultdict

print("=== Chart 2: Repositories by AI Tool Association (Before Filtering) ===")

# Logic: If a repository has ANY artifact from a known tool, associate repo with that tool
# If ALL artifacts are 'unknown', mark the repository as 'unknown'

def get_repo_tools(df):
    """
    For each repository, determine which AI tools it's associated with.
    A repo is associated with a tool if it has at least one artifact from that tool.
    If all artifacts are 'unknown', the repo is marked as 'unknown'.
    """
    repo_tool_mapping = {}
    
    for repo_name, group in df.groupby('repo_name'):
        tools = group['tool_name'].unique()
        # Filter out 'unknown' to get actual tools
        known_tools = [t for t in tools if t != 'unknown']
        
        if known_tools:
            # Repo has at least one known tool
            repo_tool_mapping[repo_name] = known_tools
        else:
            # All artifacts are unknown
            repo_tool_mapping[repo_name] = ['unknown']
    
    return repo_tool_mapping

repo_tools = get_repo_tools(original_metadata)

# Count repos per tool (a repo can be counted in multiple tools)
tool_repo_counts = defaultdict(int)
for repo, tools in repo_tools.items():
    for tool in tools:
        tool_repo_counts[tool] += 1

# Convert to DataFrame for plotting
tool_counts_df = pd.DataFrame([
    {'tool': tool, 'repo_count': count}
    for tool, count in tool_repo_counts.items()
]).sort_values('repo_count', ascending=True)

print(f"\nRepositories by AI Tool:")
for _, row in tool_counts_df.sort_values('repo_count', ascending=False).iterrows():
    pct = row['repo_count'] / len(repo_tools) * 100
    print(f"  {row['tool']}: {row['repo_count']:,} repos ({pct:.1f}%)")

# Create horizontal bar chart
fig2 = go.Figure(go.Bar(
    y=tool_counts_df['tool'],
    x=tool_counts_df['repo_count'],
    orientation='h',
    text=tool_counts_df['repo_count'],
    textposition='outside',
    marker_color=['#EF553B' if t == 'unknown' else '#00CC96' for t in tool_counts_df['tool']]
))

fig2.update_layout(
    title="Repositories by AI Tool Association<br><sup>A repository is associated with a tool if it has at least one artifact from that tool</sup>",
    xaxis_title="Number of Repositories",
    yaxis_title="AI Tool",
    height=500,
    showlegend=False
)

fig2.show()

# Additional breakdown: repos with known tools vs unknown only
repos_with_known_tools = sum(1 for tools in repo_tools.values() if tools != ['unknown'])
repos_unknown_only = sum(1 for tools in repo_tools.values() if tools == ['unknown'])

print(f"\nSummary:")
print(f"  Repos with at least one known AI tool: {repos_with_known_tools:,} ({repos_with_known_tools/len(repo_tools)*100:.1f}%)")
print(f"  Repos with only unknown artifacts: {repos_unknown_only:,} ({repos_unknown_only/len(repo_tools)*100:.1f}%)")

---
## 3. Phase 1a: Empty Content Filter

Remove artifacts with zero vectors, NaN values, or missing embeddings.

In [ ]:
print("=== Phase 1a: Empty Content Filter ===")

# Track counts before filtering
before_count = len(metadata)

# Identify invalid embeddings
has_nan = np.any(np.isnan(embeddings), axis=1)
is_zero_vector = np.all(embeddings == 0, axis=1)
is_invalid = has_nan | is_zero_vector

print(f"Artifacts with NaN values: {np.sum(has_nan):,}")
print(f"Artifacts with zero vectors: {np.sum(is_zero_vector):,}")
print(f"Total invalid: {np.sum(is_invalid):,}")

In [ ]:
# Track repo count before filtering
before_repo_count = metadata['repo_name'].nunique()

# Apply filter
valid_mask = ~is_invalid
embeddings = embeddings[valid_mask]
metadata = metadata[valid_mask].reset_index(drop=True)

after_count = len(metadata)
after_repo_count = metadata['repo_name'].nunique()
removed_count = before_count - after_count

print(f"\nRemoved {removed_count:,} artifacts with empty/invalid embeddings")
print(f"Remaining: {after_count:,} artifacts in {after_repo_count:,} repos")

# Store in report
filter_report['phases']['1a_empty_content'] = {
    'before': before_count,
    'removed': removed_count,
    'after': after_count,
    'before_repos': before_repo_count,
    'after_repos': after_repo_count,
    'removed_repos': before_repo_count - after_repo_count
}

In [ ]:
# Visualization: Before/After
fig = create_before_after_chart(
    before_count, after_count,
    "Phase 1a: Empty Content Filter Impact"
)
fig.show()

---
## 4. Phase 1b: Single-Artifact Repository Filter

Remove ALL repositories that contain only 1 artifact. These are unlikely to represent meaningful AI tool usage patterns.

In [ ]:
print("=== Phase 1b: Single-Artifact Repository Filter ===")

# Track counts before filtering
before_artifact_count = len(metadata)
before_repo_count = metadata['repo_name'].nunique()

# Count artifacts per repository
repo_artifact_counts = metadata.groupby('repo_name').size()
single_artifact_repos = repo_artifact_counts[repo_artifact_counts == 1].index

print(f"Total repositories: {before_repo_count:,}")
print(f"Single-artifact repositories: {len(single_artifact_repos):,}")
print(f"Percentage: {len(single_artifact_repos) / before_repo_count * 100:.1f}%")

In [ ]:
# Analyze artifact types in single-artifact repos
single_repo_artifacts = metadata[metadata['repo_name'].isin(single_artifact_repos)]
artifact_type_breakdown = single_repo_artifacts['artifact_name'].str.lower().value_counts().head(10)

print("\nArtifact types in single-artifact repositories:")
print(artifact_type_breakdown)

In [ ]:
# Visualization 1: Pie chart of artifact types in single-artifact repos
fig_pie = px.pie(
    values=artifact_type_breakdown.values,
    names=artifact_type_breakdown.index,
    title="Artifact Types in Single-Artifact Repositories (Top 10)",
    hole=0.3
)
fig_pie.update_traces(textposition='inside', textinfo='percent+label')
fig_pie.show()

In [ ]:
# Apply filter if enabled
if FILTER_SINGLE_ARTIFACT_REPOS:
    # Get mask for artifacts NOT in single-artifact repos
    keep_mask = ~metadata['repo_name'].isin(single_artifact_repos)
    metadata = metadata[keep_mask].reset_index(drop=True)
    embeddings = embeddings[keep_mask.values]
    
after_artifact_count = len(metadata)
after_repo_count = metadata['repo_name'].nunique()
removed_artifacts = before_artifact_count - after_artifact_count
removed_repos = before_repo_count - after_repo_count

print(f"\nFilter {'applied' if FILTER_SINGLE_ARTIFACT_REPOS else 'disabled'}")
print(f"Removed {removed_repos:,} single-artifact repositories")
print(f"Removed {removed_artifacts:,} artifacts")
print(f"Remaining: {after_artifact_count:,} artifacts in {after_repo_count:,} repos")

# Store in report
filter_report['phases']['1b_single_artifact_repo'] = {
    'before': before_artifact_count,
    'removed': removed_artifacts,
    'after': after_artifact_count,
    'before_repos': before_repo_count,
    'after_repos': after_repo_count,
    'removed_repos': removed_repos,
    'artifact_types_breakdown': artifact_type_breakdown.to_dict()
}

In [ ]:
# Verify alignment after Phase 1b
print(f"Embeddings shape after Phase 1b: {embeddings.shape}")
print(f"Metadata rows: {len(metadata)}")
assert len(embeddings) == len(metadata), "Embedding/metadata mismatch after Phase 1b"
print("✓ Embeddings and metadata are aligned")

In [ ]:
# Visualization 2: Before/After comparison
fig = make_subplots(rows=1, cols=2, subplot_titles=("Repositories", "Artifacts"))

# Repos
fig.add_trace(
    go.Bar(x=["Before", "After"], y=[before_repo_count, after_repo_count],
           text=[f"{before_repo_count:,}", f"{after_repo_count:,}"],
           textposition='outside', marker_color=['#636EFA', '#00CC96']),
    row=1, col=1
)

# Artifacts
fig.add_trace(
    go.Bar(x=["Before", "After"], y=[before_artifact_count, after_artifact_count],
           text=[f"{before_artifact_count:,}", f"{after_artifact_count:,}"],
           textposition='outside', marker_color=['#636EFA', '#00CC96']),
    row=1, col=2
)

fig.update_layout(
    title="Phase 1b: Single-Artifact Repository Filter Impact",
    showlegend=False,
    height=400
)
fig.show()

---
## 5. Phase 1c: README and License Artifacts Filter

Remove artifacts matching the README pattern (case-insensitive).

In [ ]:
print("=== Phase 1c: README Artifact Filter ===")
print(f"Pattern: {FILTER_FILE_PATTERN}")

# Track counts before filtering
before_count = len(metadata)
before_repo_count = metadata['repo_name'].nunique()
before_unique_types = metadata['artifact_name'].nunique()

# Create case-insensitive pattern
readme_pattern = re.compile(FILTER_FILE_PATTERN, re.IGNORECASE)

# Identify README artifacts
is_readme = metadata['artifact_name'].str.match(readme_pattern, na=False)

print(f"\nREADME artifacts found: {is_readme.sum():,}")
print(f"Percentage of total: {is_readme.sum() / len(metadata) * 100:.1f}%")

In [ ]:
# Analyze repository composition: boilerplate-only vs mixed content
repo_stats = metadata.groupby('repo_name').apply(
    lambda x: pd.Series({
        'total_artifacts': len(x),
        'boilerplate_count': x['artifact_name'].str.match(readme_pattern, na=False).sum()
    })
).reset_index()

repo_stats['non_boilerplate_count'] = repo_stats['total_artifacts'] - repo_stats['boilerplate_count']

# Categorize repositories
def categorize_repo(row):
    if row['boilerplate_count'] == row['total_artifacts']:
        return 'Only Boilerplate'
    elif row['boilerplate_count'] == 0:
        return 'No Boilerplate'
    else:
        return 'Mixed Content'

repo_stats['category'] = repo_stats.apply(categorize_repo, axis=1)
category_counts = repo_stats['category'].value_counts()

# Create pie chart
fig = go.Figure(data=[go.Pie(
    labels=category_counts.index,
    values=category_counts.values,
    hole=0.4,
    marker_colors=['#EF553B', '#636EFA', '#00CC96'],
    textinfo='label+percent',
    textposition='outside',
    hovertemplate='%{label}<br>Repositories: %{value}<br>Percentage: %{percent}<extra></extra>'
)])

fig.update_layout(
    title='Repository Composition: Boilerplate vs Non-Boilerplate Artifacts',
    annotations=[dict(text=f'{len(repo_stats)}<br>repos', x=0.5, y=0.5, font_size=16, showarrow=False)],
    showlegend=True,
    width=600,
    height=450
)

fig.show()

# Print summary
print(f"\nRepository breakdown:")
for cat in ['Only Boilerplate', 'Mixed Content', 'No Boilerplate']:
    if cat in category_counts:
        print(f"  {cat}: {category_counts[cat]:,} repos ({category_counts[cat]/len(repo_stats)*100:.1f}%)")
print(f"\nNote: 'Only Boilerplate' repos will be removed when filtering is applied.")

In [ ]:
# Show README variants
readme_artifacts = metadata[is_readme]['artifact_name'].value_counts()
print("\nREADME variants found:")
print(readme_artifacts.head(10))


In [ ]:
# Visualization 1: Top artifact types before/after (excluding READMEs)
artifact_counts_before = metadata['artifact_name'].str.lower().value_counts().head(15)

# Get counts without READMEs
non_readme_license_df = metadata[~is_readme]
artifact_counts_after = non_readme_license_df['artifact_name'].str.lower().value_counts().head(15)

# Create comparison chart
fig = make_subplots(rows=1, cols=2, subplot_titles=(
    "Before README and LICENSE Filter (Top 15)", 
    "After README and LICENSE Filter (Top 15)"
))

fig.add_trace(
    go.Bar(y=artifact_counts_before.index, x=artifact_counts_before.values,
           orientation='h', marker_color='#636EFA'),
    row=1, col=1
)

fig.add_trace(
    go.Bar(y=artifact_counts_after.index, x=artifact_counts_after.values,
           orientation='h', marker_color='#00CC96'),
    row=1, col=2
)

fig.update_layout(
    title="Phase 1c: Artifact Type Distribution",
    showlegend=False,
    height=500
)
fig.update_yaxes(autorange="reversed")
fig.show()

In [ ]:
# Apply filter if enabled
if FILTER_READMES:
    is_protected = metadata['artifact_name'].apply(is_protected_artifact)
    # Keep track of indices to remove (but protect AI artifacts)
    keep_mask = ~is_readme | is_protected
    print(f"Protected AI artifacts: {is_protected.sum()}")
    metadata = metadata[keep_mask].reset_index(drop=True)
    embeddings = embeddings[keep_mask.values]

after_count = len(metadata)
after_repo_count = metadata['repo_name'].nunique()
after_unique_types = metadata['artifact_name'].nunique()
removed_count = before_count - after_count

print(f"\nFilter {'applied' if FILTER_READMES else 'disabled'}")
print(f"Removed {removed_count:,} README artifacts matching pattern '{FILTER_FILE_PATTERN}'")
print(f"Remaining: {after_count:,} artifacts in {after_repo_count:,} repos")
print(f"Unique artifact types: {before_unique_types:,} -> {after_unique_types:,}")

# Store in report
filter_report['phases']['1c_readme_filter'] = {
    'before': before_count,
    'removed': removed_count,
    'after': after_count,
    'before_repos': before_repo_count,
    'after_repos': after_repo_count,
    'removed_repos': before_repo_count - after_repo_count,
    'pattern': FILTER_FILE_PATTERN
}

In [ ]:
# Visualization 2: Before/After counts
fig = create_before_after_chart(
    before_count, after_count,
    "Phase 1c: README Artifact Filter Impact"
)
fig.show()

---
## 6. Phase 1d: Repository Size Outlier Analysis

Analyze and optionally filter repositories with unusually high artifact counts.

In [ ]:
print("=== Phase 1d: Repository Size Outlier Analysis ===")
print(f"Method: {REPO_OUTLIER_METHOD}")

# Track counts before filtering
before_count = len(metadata)
before_repo_count = metadata['repo_name'].nunique()

# Calculate artifacts per repository
repo_sizes = metadata.groupby('repo_name').size()

# Calculate IQR-based thresholds
q1 = repo_sizes.quantile(0.25)
q3 = repo_sizes.quantile(0.75)
iqr = q3 - q1
iqr_threshold = q3 + REPO_OUTLIER_IQR_MULTIPLIER * iqr
extreme_threshold = q3 + 3 * iqr  # Standard extreme outlier threshold

# Calculate statistics
stats = {
    'mean': repo_sizes.mean(),
    'median': repo_sizes.median(),
    'std': repo_sizes.std(),
    'min': repo_sizes.min(),
    'max': repo_sizes.max(),
    'q1': q1,
    'q3': q3,
    'iqr': iqr,
    'iqr_threshold': iqr_threshold,
    'extreme_threshold': extreme_threshold,
    'p90': repo_sizes.quantile(0.90),
    'p95': repo_sizes.quantile(0.95),
    'p99': repo_sizes.quantile(0.99)
}

print(f"\nRepository Size Statistics:")
print(f"  Total repos: {len(repo_sizes):,}")
print(f"  Mean: {stats['mean']:.1f}")
print(f"  Median: {stats['median']:.1f}")
print(f"  Std: {stats['std']:.1f}")
print(f"  Range: {stats['min']:.0f} - {stats['max']:.0f}")
print(f"  IQR: {stats['q1']:.0f} - {stats['q3']:.0f}")
print(f"  IQR Threshold (x{REPO_OUTLIER_IQR_MULTIPLIER}): {stats['iqr_threshold']:.0f}")
print(f"  90th percentile: {stats['p90']:.0f}")
print(f"  95th percentile: {stats['p95']:.0f}")
print(f"  99th percentile: {stats['p99']:.0f}")

In [ ]:
# Visualization 1: Histogram of repository sizes
fig = go.Figure()

# Add histogram
fig.add_trace(go.Histogram(
    x=repo_sizes.values,
    nbinsx=50,
    name="Repository Size Distribution",
    marker_color='#636EFA'
))

# Add vertical lines for thresholds
fig.add_vline(x=stats['median'], line_dash="solid", line_color="green",
              annotation_text=f"Median: {stats['median']:.0f}")
fig.add_vline(x=stats['iqr_threshold'], line_dash="dash", line_color="orange",
              annotation_text=f"IQR Threshold: {stats['iqr_threshold']:.0f}")
fig.add_vline(x=stats['extreme_threshold'], line_dash="dash", line_color="red",
              annotation_text=f"Extreme: {stats['extreme_threshold']:.0f}")
fig.add_vline(x=stats['p95'], line_dash="dot", line_color="purple",
              annotation_text=f"95th %ile: {stats['p95']:.0f}")

fig.update_layout(
    title="Phase 1d: Repository Size Distribution",
    xaxis_title="Artifacts per Repository",
    yaxis_title="Number of Repositories",
    height=500
)
fig.show()

In [ ]:
# Visualization 1b: Same histogram with log scale for skewed distribution
fig_log = go.Figure()

fig_log.add_trace(go.Histogram(
    x=repo_sizes.values,
    nbinsx=50,
    name="Repository Size Distribution",
    marker_color='#636EFA'
))

fig_log.update_layout(
    title="Phase 1d: Repository Size Distribution (Log Scale)",
    xaxis_title="Artifacts per Repository",
    yaxis_title="Number of Repositories (log)",
    yaxis_type="log",
    height=500
)
fig_log.show()

In [ ]:
# Visualization 2: Top 20 repositories by artifact count
top_repos = repo_sizes.nlargest(20)

fig = go.Figure(go.Bar(
    y=top_repos.index,
    x=top_repos.values,
    orientation='h',
    marker_color='#EF553B',
    text=top_repos.values,
    textposition='outside'
))

fig.update_layout(
    title="Top 20 Repositories by Artifact Count",
    xaxis_title="Number of Artifacts",
    yaxis_title="Repository",
    height=600
)
fig.update_yaxes(autorange="reversed")
fig.show()

In [ ]:
# Visualization 3: Box plot
fig = go.Figure(go.Box(
    y=repo_sizes.values,
    name="Artifacts per Repo",
    boxpoints='outliers',
    marker_color='#636EFA'
))

fig.update_layout(
    title="Repository Size Distribution - Box Plot",
    yaxis_title="Artifacts per Repository",
    height=500
)
fig.show()

In [ ]:
# Apply filter based on method
repos_to_remove = set()

if REPO_OUTLIER_METHOD == 'iqr':
    threshold = stats['iqr_threshold']
    repos_to_remove = set(repo_sizes[repo_sizes > threshold].index)
    print(f"IQR method: removing repos with > {threshold:.0f} artifacts")
    
elif REPO_OUTLIER_METHOD == 'percentile':
    threshold = repo_sizes.quantile(REPO_OUTLIER_PERCENTILE / 100)
    repos_to_remove = set(repo_sizes[repo_sizes > threshold].index)
    print(f"Percentile method: removing repos above {REPO_OUTLIER_PERCENTILE}th percentile (>{threshold:.0f} artifacts)")
    
elif REPO_OUTLIER_METHOD == 'manual':
    repos_to_remove = set(repo_sizes[repo_sizes > REPO_OUTLIER_MAX_ARTIFACTS].index)
    print(f"Manual method: removing repos with > {REPO_OUTLIER_MAX_ARTIFACTS} artifacts")
    
else:  # 'none'
    print("No filtering applied (method='none')")

print(f"Repositories to remove: {len(repos_to_remove)}")

In [ ]:
# Apply the filter
if repos_to_remove:
    keep_mask = ~metadata['repo_name'].isin(repos_to_remove)
    metadata = metadata[keep_mask].reset_index(drop=True)
    embeddings = embeddings[keep_mask.values]

after_count = len(metadata)
after_repo_count = metadata['repo_name'].nunique()
removed_count = before_count - after_count

print(f"\nRemoved {removed_count:,} artifacts from {len(repos_to_remove)} repositories")
print(f"Remaining: {after_count:,} artifacts in {after_repo_count:,} repos")

# Store in report
filter_report['phases']['1d_repo_size_outlier'] = {
    'before': before_count,
    'removed': removed_count,
    'after': after_count,
    'before_repos': before_repo_count,
    'after_repos': after_repo_count,
    'removed_repos': len(repos_to_remove) if repos_to_remove else 0,
    'method': REPO_OUTLIER_METHOD,
    'statistics': {k: float(v) for k, v in stats.items()}
}

---
## 7. Phase 1e: LOF Embedding Outlier Filter

Use Local Outlier Factor to identify and remove embedding outliers.

In [ ]:
from collections import defaultdict
import hashlib

def embedding_to_hash(emb):
    """Convert embedding to hashable string for grouping."""
    return hashlib.md5(emb.tobytes()).hexdigest()

# Group artifacts by embedding hash
embedding_groups = defaultdict(list)
for idx, emb in enumerate(embeddings):
    emb_hash = embedding_to_hash(emb)
    embedding_groups[emb_hash].append(idx)

# Filter to only groups with duplicates (more than 1 artifact)
duplicate_groups = {h: indices for h, indices in embedding_groups.items() if len(indices) > 1}

print(f"=== Duplicate Embedding Analysis ===")
print(f"Total embeddings: {len(embeddings)}")
print(f"Unique embeddings: {len(embedding_groups)}")
print(f"Duplicate groups: {len(duplicate_groups)}")
print(f"Total duplicated artifacts: {sum(len(v) for v in duplicate_groups.values())}")

# Display duplicate groups sorted by size (largest first)
print(f"\n=== Duplicate Groups (sorted by size) ===")
for rank, (emb_hash, indices) in enumerate(sorted(duplicate_groups.items(), key=lambda x: -len(x[1])), 1):
    print(f"\n--- Group {rank}: {len(indices)} identical embeddings ---")
    for idx in indices[:10]:  # Show first 10 per group
        artifact = metadata.iloc[idx]
        # Adjust column names based on your metadata structure
        name = artifact.get('artifact_name', artifact.get('file_name', f'index_{idx}'))
        repo = artifact.get('repo_name', artifact.get('repository', 'unknown'))
        print(f"  - {repo}: {name}")
    if len(indices) > 10:
        print(f"  ... and {len(indices) - 10} more")

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 1. Histogram: Number of duplicates per embedding group
group_sizes = [len(indices) for indices in embedding_groups.values()]
duplicate_sizes = [s for s in group_sizes if s > 1]

ax1 = axes[0]
ax1.hist(duplicate_sizes, bins=range(1, max(duplicate_sizes) + 2), edgecolor='black', align='left')
ax1.set_xlabel('Group Size (# of identical embeddings)')
ax1.set_ylabel('Number of Groups')
ax1.set_title(f'Distribution of Duplicate Group Sizes\n({len(duplicate_sizes)} groups with duplicates)')
ax1.set_xticks(range(2, min(max(duplicate_sizes) + 1, 20)))

# 2. Bar chart: Duplicates per repository
repo_duplicate_counts = defaultdict(int)
repo_total_counts = defaultdict(int)

for emb_hash, indices in embedding_groups.items():
    for idx in indices:
        artifact = metadata.iloc[idx]
        repo = artifact.get('repo_name', artifact.get('repository', 'unknown'))
        repo_total_counts[repo] += 1
        if len(indices) > 1:
            repo_duplicate_counts[repo] += 1

# Sort by duplicate count, show top 20
top_repos = sorted(repo_duplicate_counts.items(), key=lambda x: -x[1])[:20]
if top_repos:
    repos, counts = zip(*top_repos)
    ax2 = axes[1]
    ax2.barh(range(len(repos)), counts, color='coral', edgecolor='black')
    ax2.set_yticks(range(len(repos)))
    ax2.set_yticklabels([r[:30] + '...' if len(r) > 30 else r for r in repos], fontsize=8)
    ax2.set_xlabel('Number of Duplicated Artifacts')
    ax2.set_title('Top 20 Repos by Duplicate Count')
    ax2.invert_yaxis()

# 3. Pie chart: Unique vs Duplicated artifacts
unique_count = sum(1 for s in group_sizes if s == 1)
duplicated_count = sum(s for s in group_sizes if s > 1)

ax3 = axes[2]
ax3.pie(
    [unique_count, duplicated_count],
    labels=[f'Unique\n({unique_count})', f'Duplicated\n({duplicated_count})'],
    autopct='%1.1f%%',
    colors=['#66b3ff', '#ff9999'],
    explode=(0, 0.05)
)
ax3.set_title('Unique vs Duplicated Embeddings')

plt.tight_layout()
plt.show()

# Summary stats
print(f"\n=== Summary ===")
print(f"Total artifacts: {len(embeddings)}")
print(f"Unique embeddings: {unique_count} ({100*unique_count/len(embeddings):.1f}%)")
print(f"Duplicated artifacts: {duplicated_count} ({100*duplicated_count/len(embeddings):.1f}%)")
print(f"Largest duplicate group: {max(group_sizes)} identical embeddings")

In [ ]:
# === Duplicate Analysis: Cross-repo vs Intra-repo ===                              
from collections import defaultdict                                                 
                                                                                    
cross_repo_groups = 0      # duplicates across different repos
intra_repo_groups = 0      # duplicates within the same repo
cross_org_groups = 0       # duplicates across different orgs
intra_org_groups = 0       # duplicates within the same org

cross_repo_artifacts = 0
intra_repo_artifacts = 0

duplicate_details = []

for emb_hash, indices in embedding_groups.items():
    if len(indices) <= 1:
        continue

    repos = set()
    orgs = set()
    artifacts_info = []

    for idx in indices:
        row = metadata.iloc[idx]
        repo = row.get('repo_name', 'unknown')
        org = row.get('org_name', repo.split('/')[0] if '/' in repo else
repo.split('_')[0])
        artifact_name = row.get('artifact_name', 'unknown')
        repos.add(repo)
        orgs.add(org)
        artifacts_info.append({'repo': repo, 'org': org, 'artifact': artifact_name})

    is_cross_repo = len(repos) > 1
    is_cross_org = len(orgs) > 1

    if is_cross_repo:
        cross_repo_groups += 1
        cross_repo_artifacts += len(indices)
    else:
        intra_repo_groups += 1
        intra_repo_artifacts += len(indices)

    if is_cross_org:
        cross_org_groups += 1
    else:
        intra_org_groups += 1

    duplicate_details.append({
        'hash': emb_hash,
        'size': len(indices),
        'repos': repos,
        'orgs': orgs,
        'is_cross_repo': is_cross_repo,
        'is_cross_org': is_cross_org,
        'artifacts': artifacts_info
    })

# === Summary ===
total_dup_groups = cross_repo_groups + intra_repo_groups
print("=" * 60)
print("DUPLICATE SCOPE ANALYSIS")
print("=" * 60)

print(f"\n--- By Repository ---")
print(f"Cross-repo duplicate groups: {cross_repo_groups} ({100*cross_repo_groups/total_dup_groups:.1f}%) → {cross_repo_artifacts} artifacts")
print(f"Intra-repo duplicate groups: {intra_repo_groups} ({100*intra_repo_groups/total_dup_groups:.1f}%) → {intra_repo_artifacts} artifacts")

print(f"\n--- By Organization ---")
print(f"Cross-org duplicate groups:  {cross_org_groups} ({100*cross_org_groups/total_dup_groups:.1f}%)")
print(f"Intra-org duplicate groups:  {intra_org_groups} ({100*intra_org_groups/total_dup_groups:.1f}%)")

# === Visualization ===
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Pie: cross-repo vs intra-repo
axes[0].pie(
    [cross_repo_groups, intra_repo_groups],
    labels=[f'Cross-repo\n({cross_repo_groups})',
f'Intra-repo\n({intra_repo_groups})'],
    autopct='%1.1f%%', colors=['#ff9999', '#66b3ff'], explode=(0.05, 0)
)
axes[0].set_title('Duplicate Groups:\nCross-repo vs Intra-repo')

# 2. Pie: cross-org vs intra-org
axes[1].pie(
    [cross_org_groups, intra_org_groups],
    labels=[f'Cross-org\n({cross_org_groups})', f'Intra-org\n({intra_org_groups})'],
    autopct='%1.1f%%', colors=['#ffcc99', '#99ff99'], explode=(0.05, 0)
)
axes[1].set_title('Duplicate Groups:\nCross-org vs Intra-org')

# 3. Top cross-repo duplicate groups - show what's being shared
cross_repo_sorted = sorted(
    [d for d in duplicate_details if d['is_cross_repo']],
    key=lambda x: -x['size']
)[:15]

if cross_repo_sorted:
    labels = []
    sizes = []
    for d in cross_repo_sorted:
        artifact_names = set(a['artifact'] for a in d['artifacts'])
        label = ', '.join(list(artifact_names)[:2])
        if len(label) > 35:
            label = label[:32] + '...'
        label += f" ({len(d['repos'])} repos)"
        labels.append(label)
        sizes.append(d['size'])

    axes[2].barh(range(len(labels)), sizes, color='salmon', edgecolor='black')
    axes[2].set_yticks(range(len(labels)))
    axes[2].set_yticklabels(labels, fontsize=8)
    axes[2].set_xlabel('Number of Identical Artifacts')
    axes[2].set_title('Top 15 Cross-Repo Duplicate Groups')
    axes[2].invert_yaxis()

plt.tight_layout()
plt.show()

# === Show examples of each type ===
print(f"\n{'=' * 60}")
print("EXAMPLES: Cross-repo duplicates (shared across repos)")
print("=" * 60)
for d in sorted([x for x in duplicate_details if x['is_cross_repo']], key=lambda x:
-x['size'])[:10]:
    artifact_names = set(a['artifact'] for a in d['artifacts'])
    print(f"\n  Artifact(s): {', '.join(artifact_names)}")
    print(f"  Shared across {len(d['repos'])} repos:")
    for repo in sorted(d['repos']):
        print(f"    - {repo}")

print(f"\n{'=' * 60}")
print("EXAMPLES: Intra-repo duplicates (same content within one repo)")
print("=" * 60)
for d in sorted([x for x in duplicate_details if not x['is_cross_repo']], key=lambda
x: -x['size'])[:10]:
    repo = list(d['repos'])[0]
    artifact_names = [a['artifact'] for a in d['artifacts']]
    print(f"\n  Repo: {repo}")
    print(f"  {len(artifact_names)} artifacts with identical embeddings:")
    for name in artifact_names:
        print(f"    - {name}")

In [ ]:
# === Phase 1f: Within-Repository Deduplication ===
print("=== Phase 1f: Within-Repository Deduplication ===")

before_count = len(metadata)
before_repo_count = metadata['repo_name'].nunique()

# Group by (repo, embedding_hash) to find intra-repo duplicates
repo_embedding_groups = defaultdict(list)
for idx, emb in enumerate(embeddings):
    artifact = metadata.iloc[idx]
    repo = artifact.get('repo_name', artifact.get('repository', 'unknown'))
    emb_hash = embedding_to_hash(emb)
    repo_embedding_groups[(repo, emb_hash)].append(idx)

# Keep only first occurrence per (repo, embedding) pair
indices_to_keep = []
removed_per_repo = defaultdict(int)

for (repo, emb_hash), indices in repo_embedding_groups.items():
    indices_to_keep.append(indices[0])  # Keep first
    if len(indices) > 1:
        removed_per_repo[repo] += len(indices) - 1

indices_to_keep = sorted(indices_to_keep)

# Apply deduplication
metadata = metadata.iloc[indices_to_keep].reset_index(drop=True)
embeddings = embeddings[indices_to_keep]

after_count = len(metadata)
after_repo_count = metadata['repo_name'].nunique()
removed_count = before_count - after_count

print(f"Before: {before_count} artifacts in {before_repo_count} repos")
print(f"After:  {after_count} artifacts in {after_repo_count} repos")
print(f"Removed: {removed_count} intra-repo duplicates ({100*removed_count/before_count:.1f}%)")

# Store in report
filter_report['phases']['1f_deduplication'] = {
    'before': before_count,
    'removed': removed_count,
    'after': after_count,
    'before_repos': before_repo_count,
    'after_repos': after_repo_count,
    'removed_repos': before_repo_count - after_repo_count
}

# Show top repos by duplicates removed
if removed_per_repo:
    print(f"\nTop 10 repos by duplicates removed:")
    for repo, count in sorted(removed_per_repo.items(), key=lambda x: -x[1])[:10]:
        print(f"  {repo}: {count} removed")

In [ ]:
# === Rebuild embedding groups after deduplication ===
embedding_groups = defaultdict(list)
for idx, emb in enumerate(embeddings):
    emb_hash = embedding_to_hash(emb)
    embedding_groups[emb_hash].append(idx)

# === Duplicate Embedding Visualizations (Post-Deduplication) ===
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 1. Histogram: Number of duplicates per embedding group
group_sizes = [len(indices) for indices in embedding_groups.values()]
duplicate_sizes = [s for s in group_sizes if s > 1]

ax1 = axes[0]
if duplicate_sizes:
    ax1.hist(duplicate_sizes, bins=range(1, max(duplicate_sizes) + 2), edgecolor='black', align='left')
    ax1.set_xticks(range(2, min(max(duplicate_sizes) + 1, 20)))
else:
    ax1.text(0.5, 0.5, 'No duplicates', ha='center', va='center', transform=ax1.transAxes)
ax1.set_xlabel('Group Size (# of identical embeddings)')
ax1.set_ylabel('Number of Groups')
ax1.set_title(f'Distribution of Duplicate Group Sizes\n({len(duplicate_sizes)} groups with duplicates)')

# 2. Bar chart: Duplicates per repository (cross-repo duplicates)
repo_duplicate_counts = defaultdict(int)
repo_total_counts = defaultdict(int)

for emb_hash, indices in embedding_groups.items():
    for idx in indices:
        artifact = metadata.iloc[idx]
        repo = artifact.get('repo_name', artifact.get('repository', 'unknown'))
        repo_total_counts[repo] += 1
        if len(indices) > 1:
            repo_duplicate_counts[repo] += 1

# Sort by duplicate count, show top 20
top_repos = sorted(repo_duplicate_counts.items(), key=lambda x: -x[1])[:20]
if top_repos:
    repos, counts = zip(*top_repos)
    ax2 = axes[1]
    ax2.barh(range(len(repos)), counts, color='coral', edgecolor='black')
    ax2.set_yticks(range(len(repos)))
    ax2.set_yticklabels([r[:30] + '...' if len(r) > 30 else r for r in repos], fontsize=8)
    ax2.set_xlabel('Number of Cross-Repo Duplicated Artifacts')
    ax2.set_title('Top 20 Repos by Cross-Repo Duplicate Count')
    ax2.invert_yaxis()

# 3. Pie chart: Unique vs Duplicated artifacts
unique_count = sum(1 for s in group_sizes if s == 1)
duplicated_count = sum(s for s in group_sizes if s > 1)

ax3 = axes[2]
ax3.pie(
    [unique_count, duplicated_count],
    labels=[f'Unique\n({unique_count})', f'Duplicated\n({duplicated_count})'],
    autopct='%1.1f%%',
    colors=['#66b3ff', '#ff9999'],
    explode=(0, 0.05)
)
ax3.set_title('Unique vs Cross-Repo Duplicated Embeddings')

plt.tight_layout()
plt.show()

# Summary stats
print(f"\n=== Summary (Post Intra-Repo Deduplication) ===")
print(f"Total artifacts: {len(embeddings)}")
print(f"Unique embeddings: {unique_count} ({100*unique_count/len(embeddings):.1f}%)")
print(f"Cross-repo duplicated artifacts: {duplicated_count} ({100*duplicated_count/len(embeddings):.1f}%)")
if duplicate_sizes:
    print(f"Largest cross-repo duplicate group: {max(group_sizes)} identical embeddings")

In [ ]:
# === Explore Cross-Repo Duplicate Patterns (Full) ===
print("=== Cross-Repo Duplicate Analysis ===\n")

# Find groups with duplicates across repos
cross_repo_groups = []
for emb_hash, indices in embedding_groups.items():
    if len(indices) > 1:
        repos_in_group = set()
        artifacts_info = []
        for idx in indices:
            artifact = metadata.iloc[idx]
            repo = artifact.get('repo_name', artifact.get('repository', 'unknown'))
            name = artifact.get('artifact_name', artifact.get('file_name', f'idx_{idx}'))
            repos_in_group.add(repo)
            artifacts_info.append({'repo': repo, 'name': name, 'idx': idx})
        cross_repo_groups.append({
            'hash': emb_hash,
            'count': len(indices),
            'repo_count': len(repos_in_group),
            'repos': repos_in_group,
            'artifacts': artifacts_info
        })

# Sort by number of repos sharing the same embedding
cross_repo_groups.sort(key=lambda x: (-x['repo_count'], -x['count']))

print(f"Total cross-repo duplicate groups: {len(cross_repo_groups)}")
print(f"Groups spanning multiple repos: {sum(1 for g in cross_repo_groups if g['repo_count'] > 1)}")

# Show ALL groups that span MULTIPLE repos
print("\n=== All Cross-Repo Shared Embeddings ===")
group_num = 0
for group in cross_repo_groups:
    if group['repo_count'] > 1:  # Only show if actually across multiple repos
        group_num += 1
        print(f"\n--- Group {group_num}: Shared by {group['repo_count']} repos ({group['count']} artifacts) ---")
        for art in group['artifacts']:
            print(f"  [{art['repo']}] {art['name']}")

# Summary: groups within same repo vs across repos
same_repo_groups = sum(1 for g in cross_repo_groups if g['repo_count'] == 1)
multi_repo_groups = sum(1 for g in cross_repo_groups if g['repo_count'] > 1)
print(f"\n\n=== Summary ===")
print(f"Duplicate groups within same repo (after dedup - shouldn't exist): {same_repo_groups}")
print(f"Duplicate groups across multiple repos: {multi_repo_groups}")

# Check for same-org patterns
print("\n=== Organization Pattern Analysis ===")
org_prefixes = defaultdict(list)
for group in cross_repo_groups:
    if group['repo_count'] > 1:
        for repo in group['repos']:
            org = repo.split('_')[0] if '_' in repo else repo.split('-')[0]
            org_prefixes[org].append(group['hash'])

print("Organizations with cross-repo sharing:")
for org, hashes in sorted(org_prefixes.items(), key=lambda x: -len(set(x[1]))):
    unique_shared = len(set(hashes))
    if unique_shared > 0:
        print(f"  {org}: {unique_shared} unique shared embeddings")

In [ ]:
print("=== Phase 1e: LOF Embedding Outlier Filter ===")

# Track counts before filtering
before_count = len(metadata)
before_repo_count = metadata['repo_name'].nunique()
n_samples = len(embeddings)

# Calculate n_neighbors
if LOF_N_NEIGHBORS_AUTO:
    n_neighbors = calculate_auto_n_neighbors(n_samples)
    print(f"Auto-calculated n_neighbors: {n_neighbors}")
    print(f"  Formula: min(50, max(10, int(sqrt({n_samples}) / 2)))")
else:
    n_neighbors = LOF_N_NEIGHBORS_MANUAL
    print(f"Manual n_neighbors: {n_neighbors}")

# Generate sensitivity test values
sensitivity_values = generate_sensitivity_test_values(n_neighbors)
print(f"Sensitivity test values: {sensitivity_values}")

In [ ]:
# Sensitivity Analysis: Compute LOF scores for different n_neighbors values
print("\nRunning LOF sensitivity analysis...")

lof_scores_by_k = {}
for k in sensitivity_values:
    print(f"  Computing LOF with n_neighbors={k}...")
    lof = LocalOutlierFactor(n_neighbors=k, contamination=LOF_CONTAMINATION, novelty=False)
    lof.fit_predict(embeddings)
    # Negative outlier factor - more negative = more outlier
    lof_scores_by_k[k] = -lof.negative_outlier_factor_

print("Sensitivity analysis complete.")

In [ ]:
# Visualization 1: Score distributions for each n_neighbors value
fig = go.Figure()

for k, scores in lof_scores_by_k.items():
    fig.add_trace(go.Histogram(
        x=scores,
        name=f"k={k}",
        opacity=0.6,
        nbinsx=50
    ))

fig.update_layout(
    title="LOF Score Distributions by n_neighbors",
    xaxis_title="LOF Score (higher = more outlier)",
    yaxis_title="Count",
    barmode='overlay',
    height=500
)
fig.show()

In [ ]:
#=== Investigate Extreme Outliers ===
# Use the auto-calculated n_neighbors value
primary_k = n_neighbors
lof_scores = lof_scores_by_k[primary_k]

# Add scores to metadata temporarily for inspection
metadata['lof_score_temp'] = lof_scores

# Show top 20 outliers
print(f"=== Top 20 Outliers (k={primary_k}) ===\n")
top_outliers = metadata.nlargest(20, 'lof_score_temp')

for idx, row in top_outliers.iterrows():
    repo = row.get('repo_name', row.get('repository', 'unknown'))
    name = row.get('artifact_name', row.get('file_name', 'unknown'))
    score = row['lof_score_temp']
    size = row.get('file_size', row.get('content_length', 'N/A'))
    print(f"LOF: {score:,.0f} | Size: {size} | [{repo}] {name}")

# Clean up temp column
metadata.drop('lof_score_temp', axis=1, inplace=True)

In [ ]:
# Visualization 2: Rank correlation heatmap between n_neighbors values
from scipy.stats import spearmanr

# Create correlation matrix
n_k = len(sensitivity_values)
corr_matrix = np.zeros((n_k, n_k))

for i, k1 in enumerate(sensitivity_values):
    for j, k2 in enumerate(sensitivity_values):
        corr, _ = spearmanr(lof_scores_by_k[k1], lof_scores_by_k[k2])
        corr_matrix[i, j] = corr

fig = go.Figure(data=go.Heatmap(
    z=corr_matrix,
    x=[f"k={k}" for k in sensitivity_values],
    y=[f"k={k}" for k in sensitivity_values],
    colorscale='RdBu',
    zmid=0.5,
    text=np.round(corr_matrix, 2),
    texttemplate="%{text}",
    textfont={"size": 12}
))

fig.update_layout(
    title="Rank Correlation Between LOF Scores (Spearman)",
    height=500
)
fig.show()

In [ ]:
# Visualization 3: Stability chart - % of top outliers consistent across k values
top_pct = 0.05  # Top 5% as outliers
n_top = int(n_samples * top_pct)

# Get top outlier indices for each k
top_outliers_by_k = {}
for k, scores in lof_scores_by_k.items():
    top_indices = np.argsort(scores)[-n_top:]
    top_outliers_by_k[k] = set(top_indices)

# Calculate overlap between consecutive k values
stability_scores = []
for i in range(1, len(sensitivity_values)):
    k_prev = sensitivity_values[i-1]
    k_curr = sensitivity_values[i]
    overlap = len(top_outliers_by_k[k_prev] & top_outliers_by_k[k_curr])
    stability = overlap / n_top * 100
    stability_scores.append(stability)

fig = go.Figure(go.Bar(
    x=[f"k={sensitivity_values[i-1]} vs k={sensitivity_values[i]}" for i in range(1, len(sensitivity_values))],
    y=stability_scores,
    text=[f"{s:.1f}%" for s in stability_scores],
    textposition='outside',
    marker_color='#00CC96'
))

fig.update_layout(
    title=f"Outlier Stability: % of Top {top_pct*100:.0f}% Outliers Consistent Across k Values",
    yaxis_title="Overlap Percentage",
    height=400
)
fig.show()

In [ ]:
# Compute final LOF scores with selected n_neighbors                                
print(f"\nComputing final LOF scores with n_neighbors={n_neighbors}...")            
                
final_lof = LocalOutlierFactor(n_neighbors=n_neighbors, contamination='auto',
novelty=False)
lof_predictions = final_lof.fit_predict(embeddings)
final_scores = -final_lof.negative_outlier_factor_

# Diagnostic only — no filtering, let HDBSCAN handle noise
is_outlier = np.zeros(len(final_scores), dtype=bool)  # No outliers removed

print(f"LOF score range: [{final_scores.min():.3f}, {final_scores.max():.3f}]")
print(f"LOF median: {np.median(final_scores):.3f}, mean: {np.mean(final_scores):.3f}")
print(f"Points with LOF > 1.5: {(final_scores > 1.5).sum()} ({100*(final_scores > 1.5).sum()/len(final_scores):.1f}%)")
print(f"Points with LOF > 2.0: {(final_scores > 2.0).sum()} ({100*(final_scores > 2.0).sum()/len(final_scores):.1f}%)")
print(f"Outliers removed: 0 (diagnostic only)")


In [ ]:
# Visualization 4: LOF score distribution (diagnostic only)                         
fig = go.Figure()
                                                                                    
fig.add_trace(go.Histogram(
    x=final_scores,
    nbinsx=100,
    name="LOF Scores",
    marker_color='#636EFA'
))

fig.update_layout(
    title="LOF Score Distribution (diagnostic only — no filtering applied)",
    xaxis_title="LOF Score (higher = more outlier)",
    yaxis_title="Count",
    height=500
)
fig.show()

In [ ]:
                                                                                    
# Visualization 6: Artifact types with highest median LOF scores (diagnostic)       
metadata_with_scores = metadata.copy()                                              
metadata_with_scores['lof_score'] = final_scores                                  

# Calculate median LOF per artifact type
artifact_lof_stats = metadata_with_scores.groupby('artifact_name').agg(
    median_lof=('lof_score', 'median'),
    mean_lof=('lof_score', 'mean'),
    max_lof=('lof_score', 'max'),
    count=('lof_score', 'count')
).reset_index()

# Filter: at least 5 samples
artifact_lof_stats = artifact_lof_stats[artifact_lof_stats['count'] >= 5]
artifact_lof_stats = artifact_lof_stats.sort_values('median_lof',
ascending=False).head(15)

fig = go.Figure(go.Bar(
    y=artifact_lof_stats['artifact_name'],
    x=artifact_lof_stats['median_lof'],
    orientation='h',
    marker_color='#EF553B',
    text=[f"median={med:.2f}, n={n}" for med, n in zip(
        artifact_lof_stats['median_lof'],
        artifact_lof_stats['count'].astype(int)
    )],
    textposition='outside'
))

fig.update_layout(
    title="Artifact Types with Highest Median LOF Scores (diagnostic only)",
    xaxis_title="Median LOF Score",
    height=500
)
fig.update_yaxes(autorange="reversed")
fig.add_vline(x=1.0, line_dash="dash", line_color="gray", annotation_text="LOF=1.0 (normal)")
fig.show()

In [ ]:
# Apply whitelist protection
protected_count = 0
if PROTECT_KNOWN_AI_ARTIFACTS:
    print("\nApplying whitelist protection for known AI artifacts...")

    # Create protection mask for outliers only
    is_protected = np.zeros(len(metadata), dtype=bool)

    for idx in range(len(metadata)):
        if is_outlier[idx]:
            artifact_name = metadata.iloc[idx]['artifact_name']
            artifact_path = metadata.iloc[idx].get('artifact_path', '')
            if is_protected_artifact(artifact_name, artifact_path):
                is_protected[idx] = True
                protected_count += 1

    # Update outlier mask to exclude protected artifacts
    final_outliers = is_outlier & ~is_protected

    print(f"Protected {protected_count} known AI artifacts from removal")

    # Show which artifacts were protected
    if protected_count > 0:
        protected_artifacts = metadata[is_protected]['artifact_name'].value_counts()
        print("\nProtected artifact types:")
        for name, count in protected_artifacts.items():
            print(f"  - {name}: {count}")
else:
    final_outliers = is_outlier
    print("Whitelist protection disabled")

In [ ]:
# Apply filter if enabled
if REMOVE_LOF_OUTLIERS:
    keep_mask = ~final_outliers
    metadata = metadata[keep_mask].reset_index(drop=True)
    embeddings = embeddings[keep_mask]

after_count = len(metadata)
after_repo_count = metadata['repo_name'].nunique()
removed_count = before_count - after_count

print(f"\nFilter {'applied' if REMOVE_LOF_OUTLIERS else 'disabled'}")
print(f"Would remove: {is_outlier.sum():,}")
print(f"Protected by whitelist: {protected_count:,}")
print(f"Actually removed: {removed_count:,}")
print(f"Remaining: {after_count:,} artifacts in {after_repo_count:,} repos")

# Store in report
filter_report['phases']['1e_lof_outlier'] = {
    'before': before_count,
    'removed': removed_count,
    'after': after_count,
    'before_repos': before_repo_count,
    'after_repos': after_repo_count,
    'removed_repos': before_repo_count - after_repo_count,
    'n_neighbors': n_neighbors,
    'n_neighbors_auto': LOF_N_NEIGHBORS_AUTO,
    'sensitivity_test_values': sensitivity_values,
    'contamination': LOF_CONTAMINATION,
    'threshold_score': float(final_scores.max()) if len(final_scores) > 0 else 0.0,
    'protected_by_whitelist': protected_count
}

---
## 8. Summary Dashboard

In [ ]:
print("=== Filter Pipeline Summary ===")

# Create summary table
summary_data = []
for phase_key, phase_data in filter_report['phases'].items():
    before = phase_data['before']
    removed = phase_data['removed']
    after = phase_data['after']
    pct_removed = (removed / before * 100) if before > 0 else 0
    
    summary_data.append({
        'Phase': phase_key,
        'Before': before,
        'Removed': removed,
        'After': after,
        '% Removed': f"{pct_removed:.1f}%"
    })

# Add total row
total_before = filter_report['input']['total_artifacts']
total_after = len(metadata)
total_removed = total_before - total_after
total_pct = (total_removed / total_before * 100) if total_before > 0 else 0

summary_data.append({
    'Phase': 'TOTAL',
    'Before': total_before,
    'Removed': total_removed,
    'After': total_after,
    '% Removed': f"{total_pct:.1f}%"
})

summary_df = pd.DataFrame(summary_data)
print(summary_df.to_string(index=False))

In [ ]:
# Visualization 1: Sankey diagram of data flow
phases = list(filter_report['phases'].keys())
labels = ['Start'] + [f"After {p}" for p in phases]

# Build source, target, value for Sankey
sources = []
targets = []
values = []
colors = []

prev_count = filter_report['input']['total_artifacts']
for i, (phase_key, phase_data) in enumerate(filter_report['phases'].items()):
    # Flow from previous to current (kept)
    sources.append(i)
    targets.append(i + 1)
    values.append(phase_data['after'])
    colors.append('rgba(0, 204, 150, 0.5)')  # Green for kept
    
    # If any removed, add a "removed" node
    if phase_data['removed'] > 0:
        removed_node = len(labels)
        labels.append(f"Removed by {phase_key}")
        sources.append(i)
        targets.append(removed_node)
        values.append(phase_data['removed'])
        colors.append('rgba(239, 85, 59, 0.5)')  # Red for removed

fig = go.Figure(go.Sankey(
    node=dict(
        pad=15,
        thickness=20,
        line=dict(color="black", width=0.5),
        label=labels,
    ),
    link=dict(
        source=sources,
        target=targets,
        value=values,
        color=colors
    )
))

fig.update_layout(
    title="Data Flow Through Filter Pipeline",
    font_size=12,
    height=600
)
fig.show()

In [ ]:
# === Filtration Summary Table ===
import pandas as pd

# Define human-readable phase names and descriptions
phase_info = {
    '1a_empty_content': ('Empty Content', 'Remove artifacts with invalid embeddings'),
    '1b_single_artifact_repo': ('Single-File Repos', 'Remove repositories with only one artifact'),
    '1c_readme_filter': ('Boilerplate Files', 'Remove README, LICENSE, CHANGELOG, PR templates'),
    '1d_repo_size_outlier': ('Large Repos', 'Remove repositories with excessive artifact counts'),
    '1f_deduplication': ('Duplicates', 'Remove duplicate embeddings within repositories'),
    '1e_lof_outlier': ('Embedding Outliers', 'Remove statistically unusual artifacts (LOF)'),
}

# Get original counts from input
original_artifacts = filter_report['input']['total_artifacts']
original_repos = filter_report['input']['total_repos']

# Build summary data
summary_data = []
cumulative_removed_artifacts = 0
cumulative_removed_repos = 0

for phase_key in filter_report['phases'].keys():
    phase = filter_report['phases'][phase_key]
    
    if phase_key in phase_info:
        phase_name, description = phase_info[phase_key]
    else:
        phase_name = phase_key.replace('_', ' ').title()
        description = 'No description available'
    
    # Artifacts
    before_artifacts = phase.get('before', 0)
    after_artifacts = phase.get('after', 0)
    removed_artifacts = phase.get('removed', 0)
    pct_artifacts = (removed_artifacts / before_artifacts * 100) if before_artifacts > 0 else 0
    
    # Cumulative artifacts
    cumulative_removed_artifacts += removed_artifacts
    cumulative_pct_artifacts = (cumulative_removed_artifacts / original_artifacts * 100) if original_artifacts > 0 else 0
    
    # Repos
    before_repos = phase.get('before_repos', None)
    after_repos = phase.get('after_repos', None)
    removed_repos = phase.get('removed_repos', None)
    
    if isinstance(before_repos, int) and before_repos > 0 and isinstance(removed_repos, int):
        pct_repos = (removed_repos / before_repos * 100)
        cumulative_removed_repos += removed_repos
        cumulative_pct_repos = (cumulative_removed_repos / original_repos * 100) if original_repos > 0 else 0
    else:
        pct_repos = None
        cumulative_pct_repos = None
    
    summary_data.append({
        'Phase': phase_name,
        'Description': description,
        'Repos Before': f"{before_repos:,}" if isinstance(before_repos, int) else '-',
        'Repos After': f"{after_repos:,}" if isinstance(after_repos, int) else '-',
        'Repos Removed %': f"{pct_repos:.1f}%" if pct_repos is not None else '-',
        'Repos Cumul. %': f"{cumulative_pct_repos:.1f}%" if cumulative_pct_repos is not None else '-',
        'Artifacts Before': f"{before_artifacts:,}",
        'Artifacts After': f"{after_artifacts:,}",
        'Artifacts Removed %': f"{pct_artifacts:.1f}%",
        'Artifacts Cumul. %': f"{cumulative_pct_artifacts:.1f}%"
    })

summary_df = pd.DataFrame(summary_data)

# Plotly table
fig = go.Figure(data=[go.Table(
    header=dict(
        values=[
            '<b>Phase</b>', 
            '<b>Description</b>', 
            '<b>Repos<br>Before</b>', 
            '<b>Repos<br>After</b>', 
            '<b>Repos<br>Removed %</b>',
            '<b>Repos<br>Cumul. %</b>',
            '<b>Artifacts<br>Before</b>', 
            '<b>Artifacts<br>After</b>', 
            '<b>Artifacts<br>Removed %</b>',
            '<b>Artifacts<br>Cumul. %</b>'
        ],
        fill_color=['#636EFA', '#636EFA', '#00CC96', '#00CC96', '#00CC96', '#00CC96', '#EF553B', '#EF553B', '#EF553B', '#EF553B'],
        font=dict(color='white', size=11),
        align='center',
        height=40
    ),
    cells=dict(
        values=[
            summary_df['Phase'],
            summary_df['Description'],
            summary_df['Repos Before'],
            summary_df['Repos After'],
            summary_df['Repos Removed %'],
            summary_df['Repos Cumul. %'],
            summary_df['Artifacts Before'],
            summary_df['Artifacts After'],
            summary_df['Artifacts Removed %'],
            summary_df['Artifacts Cumul. %']
        ],
        fill_color=[['#f8f9fa', '#ffffff'] * (len(summary_df) // 2 + 1)][:len(summary_df)],
        align=['left', 'left', 'right', 'right', 'right', 'right', 'right', 'right', 'right', 'right'],
        height=30,
        font=dict(size=11)
    ),
    columnwidth=[1, 2, 0.7, 0.7, 0.7, 0.7, 0.7, 0.7, 0.7, 0.7]
)])

fig.update_layout(
    title=f"Repository Artifacts Filtration Pipeline (Original: {original_repos:,} repos, {original_artifacts:,} artifacts)",
    height=120 + len(summary_df) * 35,
    margin=dict(l=20, r=20, t=50, b=20)
)
fig.show()

# Print final summary
print(f"\n=== Final Summary ===")
print(f"Original: {original_repos:,} repos, {original_artifacts:,} artifacts")
print(f"Final: {summary_data[-1]['Repos After']} repos, {summary_data[-1]['Artifacts After']} artifacts")
print(f"Total removed: {cumulative_pct_repos:.1f}% repos, {cumulative_pct_artifacts:.1f}% artifacts")

In [ ]:
# Visualization 3: Before/after artifact distribution
# Use the original_metadata we stored at the start
original_artifact_counts = original_metadata['artifact_name'].str.lower().value_counts().head(15)
final_artifact_counts = metadata['artifact_name'].str.lower().value_counts().head(15)

fig = make_subplots(rows=1, cols=2, subplot_titles=(
    f"Before Filtering (n={len(original_metadata):,})",
    f"After Filtering (n={len(metadata):,})"
))

fig.add_trace(
    go.Bar(y=original_artifact_counts.index, x=original_artifact_counts.values,
           orientation='h', marker_color='#636EFA'),
    row=1, col=1
)

fig.add_trace(
    go.Bar(y=final_artifact_counts.index, x=final_artifact_counts.values,
           orientation='h', marker_color='#00CC96'),
    row=1, col=2
)

fig.update_layout(
    title="Artifact Type Distribution: Before vs After Filtering",
    showlegend=False,
    height=500
)
fig.update_yaxes(autorange="reversed")
fig.show()

---
## 9. Export: Two Filtered Datasets

In [ ]:
print("=== Exporting Filtered Datasets ===")

# Dataset 1: Common Filters Only (what we've built so far)
print("\n--- Dataset 1: Common Filters Only ---")

# Ensure data directory exists
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Save common filtered embeddings
np.savez_compressed(
    OUTPUT_COMMON_EMBEDDINGS,
    embeddings=embeddings,
    global_file_ids=metadata['global_file_id'].values if 'global_file_id' in metadata.columns else np.arange(len(metadata))
)
print(f"Saved: {OUTPUT_COMMON_EMBEDDINGS}")

# Save common filtered metadata
metadata.to_csv(OUTPUT_COMMON_METADATA, index=False)
print(f"Saved: {OUTPUT_COMMON_METADATA}")

print(f"\nDataset 1 Statistics:")
print(f"  Artifacts: {len(metadata):,}")
print(f"  Repositories: {metadata['repo_name'].nunique():,}")

In [ ]:
# Dataset 2: Common + AI Tools Filter
print("\n--- Dataset 2: Common + AI Tools Filter ---")
print("Keeping only repos with at least one known AI artifact...")

# Identify repos with known AI artifacts
repos_with_ai_artifacts = set()

for idx, row in metadata.iterrows():
    artifact_name = row['artifact_name']
    artifact_path = row.get('artifact_path', '')
    if is_protected_artifact(artifact_name, artifact_path):
        repos_with_ai_artifacts.add(row['repo_name'])

print(f"Repositories with known AI artifacts: {len(repos_with_ai_artifacts):,}")

# Show breakdown of AI artifacts found
ai_artifact_counts = {}
for idx, row in metadata.iterrows():
    artifact_name = row['artifact_name']
    artifact_path = row.get('artifact_path', '')
    if is_protected_artifact(artifact_name, artifact_path):
        ai_artifact_counts[artifact_name] = ai_artifact_counts.get(artifact_name, 0) + 1

print(f"\nAI artifact types found:")
for name, count in sorted(ai_artifact_counts.items(), key=lambda x: -x[1])[:15]:
    print(f"  - {name}: {count}")

In [ ]:
# Apply AI tools filter
ai_filtered_mask = metadata['repo_name'].isin(repos_with_ai_artifacts)
ai_metadata = metadata[ai_filtered_mask].reset_index(drop=True)
ai_embeddings = embeddings[ai_filtered_mask.values]

removed_by_ai_filter = len(metadata) - len(ai_metadata)

print(f"\nAI Tools Filter Results:")
print(f"  Before: {len(metadata):,} artifacts")
print(f"  After: {len(ai_metadata):,} artifacts")
print(f"  Removed: {removed_by_ai_filter:,} artifacts")

In [ ]:
# Visualization: Dataset 1 vs Dataset 2 comparison
fig = make_subplots(rows=1, cols=2, subplot_titles=(
    "Artifact Count",
    "Repository Count"
))

# Artifact counts
fig.add_trace(
    go.Bar(
        x=["Common Filters", "+ AI Tools Filter"],
        y=[len(metadata), len(ai_metadata)],
        text=[f"{len(metadata):,}", f"{len(ai_metadata):,}"],
        textposition='outside',
        marker_color=['#636EFA', '#00CC96']
    ),
    row=1, col=1
)

# Repository counts
fig.add_trace(
    go.Bar(
        x=["Common Filters", "+ AI Tools Filter"],
        y=[metadata['repo_name'].nunique(), ai_metadata['repo_name'].nunique()],
        text=[f"{metadata['repo_name'].nunique():,}", f"{ai_metadata['repo_name'].nunique():,}"],
        textposition='outside',
        marker_color=['#636EFA', '#00CC96']
    ),
    row=1, col=2
)

fig.update_layout(
    title="Dataset 1 vs Dataset 2 Comparison",
    showlegend=False,
    height=400
)
fig.show()

In [ ]:
# === Repository Distribution Analysis ===
from plotly.subplots import make_subplots

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        "Repos by Artifact Count",
        "Repos by AI Tool"
    ),
    horizontal_spacing=0.15
)

# --- Chart 1: Distribution of repositories by number of artifacts (Combined) ---

# Common Filter
repo_artifact_counts_common = metadata.groupby('repo_name').size()
common_hist = repo_artifact_counts_common.value_counts().sort_index()

# AI Tools Only
repo_artifact_counts_ai = ai_metadata.groupby('repo_name').size()
ai_hist = repo_artifact_counts_ai.value_counts().sort_index()

# Get all unique artifact counts for x-axis
all_counts = sorted(set(common_hist.index[:20]) | set(ai_hist.index[:20]))

fig.add_trace(
    go.Bar(
        x=[str(x) for x in all_counts],
        y=[common_hist.get(x, 0) for x in all_counts],
        marker_color='#636EFA',
        name='Common Filter'
    ),
    row=1, col=1
)

fig.add_trace(
    go.Bar(
        x=[str(x) for x in all_counts],
        y=[ai_hist.get(x, 0) for x in all_counts],
        marker_color='#00CC96',
        name='AI Tools Only'
    ),
    row=1, col=1
)

# --- Chart 2: Repositories by AI tool association (Combined) ---

# Common Filter - show ALL tools including unknown
tool_repo_common = metadata.groupby('tool_name')['repo_name'].nunique().sort_values(ascending=False)

# AI Tools Only - exclude unknown
known_tools_only = ai_metadata[ai_metadata['tool_name'] != 'unknown']
tool_repo_ai = known_tools_only.groupby('tool_name')['repo_name'].nunique()

# Get all unique tools
all_tools = tool_repo_common.head(10).index.tolist()

fig.add_trace(
    go.Bar(
        y=all_tools,
        x=[tool_repo_common.get(t, 0) for t in all_tools],
        orientation='h',
        marker_color='#636EFA',
        name='Common Filter',
        showlegend=False
    ),
    row=1, col=2
)

fig.add_trace(
    go.Bar(
        y=all_tools,
        x=[tool_repo_ai.get(t, 0) for t in all_tools],
        orientation='h',
        marker_color='#00CC96',
        name='AI Tools Only',
        showlegend=False
    ),
    row=1, col=2
)

# Update layout
fig.update_layout(
    title="Repository Distribution Analysis: Common Filter vs AI Tools Only",
    barmode='group',
    height=500,
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="center",
        x=0.5
    )
)

# Update axes labels
fig.update_xaxes(title_text="Artifacts per Repo", row=1, col=1)
fig.update_yaxes(title_text="Number of Repos", row=1, col=1)
fig.update_xaxes(title_text="Number of Repos", row=1, col=2)
fig.update_yaxes(autorange="reversed", row=1, col=2)

fig.show()

# Print summary stats
print("\n=== Summary Statistics ===")
print(f"\nCommon Filter:")
print(f"  Repos: {metadata['repo_name'].nunique():,}")
print(f"  Artifacts: {len(metadata):,}")
print(f"  Avg artifacts/repo: {len(metadata)/metadata['repo_name'].nunique():.1f}")

print(f"\nAI Tools Only:")
print(f"  Repos: {ai_metadata['repo_name'].nunique():,}")
print(f"  Artifacts: {len(ai_metadata):,}")
print(f"  Avg artifacts/repo: {len(ai_metadata)/ai_metadata['repo_name'].nunique():.1f}")

In [ ]:
# Save AI tools filtered dataset
np.savez_compressed(
    OUTPUT_AI_TOOLS_EMBEDDINGS,
    embeddings=ai_embeddings,
    global_file_ids=ai_metadata['global_file_id'].values if 'global_file_id' in ai_metadata.columns else np.arange(len(ai_metadata))
)
print(f"Saved: {OUTPUT_AI_TOOLS_EMBEDDINGS}")

ai_metadata.to_csv(OUTPUT_AI_TOOLS_METADATA, index=False)
print(f"Saved: {OUTPUT_AI_TOOLS_METADATA}")

print(f"\nDataset 2 Statistics:")
print(f"  Artifacts: {len(ai_metadata):,}")
print(f"  Repositories: {ai_metadata['repo_name'].nunique():,}")

In [ ]:
# === Verify Data Integrity Before Export ===
print("=== Data Integrity Check ===\n")

# Check 1: Alignment
print("1. Alignment Check:")
print(f"   Dataset 1 - embeddings: {len(embeddings)}, metadata: {len(metadata)}")
assert len(embeddings) == len(metadata), "❌ Dataset 1: Embeddings/metadata mismatch!"
print(f"   ✓ Dataset 1 aligned")

print(f"   Dataset 2 - embeddings: {len(ai_embeddings)}, metadata: {len(ai_metadata)}")
assert len(ai_embeddings) == len(ai_metadata), "❌ Dataset 2: Embeddings/metadata mismatch!"
print(f"   ✓ Dataset 2 aligned")

# Check 2: Dataset 2 is subset of Dataset 1
print("\n2. Subset Check:")
assert len(ai_metadata) <= len(metadata), "❌ Dataset 2 larger than Dataset 1!"
print(f"   ✓ Dataset 2 ({len(ai_metadata):,}) ⊆ Dataset 1 ({len(metadata):,})")

# Check 3: All AI repos have known AI artifacts
print("\n3. AI Tools Filter Integrity:")
ai_repos_in_dataset = set(ai_metadata['repo_name'].unique())
print(f"   AI repos in dataset: {len(ai_repos_in_dataset):,}")
print(f"   AI repos from filter: {len(repos_with_ai_artifacts):,}")
assert ai_repos_in_dataset == repos_with_ai_artifacts, "❌ Repo mismatch!"
print(f"   ✓ All AI repos correctly filtered")

# Check 4: Embedding dimensions
print("\n4. Embedding Dimensions:")
print(f"   Dataset 1: {embeddings.shape}")
print(f"   Dataset 2: {ai_embeddings.shape}")
assert embeddings.shape[1] == ai_embeddings.shape[1], "❌ Embedding dimension mismatch!"
print(f"   ✓ Same embedding dimension ({embeddings.shape[1]})")

# Check 5: Required columns exist
print("\n5. Required Columns:")
required_cols = ['repo_name', 'artifact_name', 'tool_name']
for col in required_cols:
    assert col in metadata.columns, f"❌ Missing {col} in metadata"
    assert col in ai_metadata.columns, f"❌ Missing {col} in ai_metadata"
print(f"   ✓ All required columns present: {required_cols}")

# Check 6: No NaN in critical columns
print("\n6. NaN Check:")
for col in required_cols:
    nan_count_1 = metadata[col].isna().sum()
    nan_count_2 = ai_metadata[col].isna().sum()
    if nan_count_1 > 0 or nan_count_2 > 0:
        print(f"   ⚠️  {col}: Dataset1={nan_count_1}, Dataset2={nan_count_2} NaN values")
    else:
        print(f"   ✓ {col}: No NaN values")

print("\n=== All Checks Passed ✓ ===")

In [ ]:
# Complete the filter report
filter_report['output'] = {
    'common': {
        'artifacts': len(metadata),
        'repos': metadata['repo_name'].nunique(),
        'tool_breakdown': metadata['tool_name'].value_counts().to_dict()
    },
    'ai_tools': {
        'artifacts': len(ai_metadata),
        'repos': ai_metadata['repo_name'].nunique(),
        'removed_by_ai_filter': removed_by_ai_filter,
        'tool_breakdown': ai_metadata[ai_metadata['tool_name'] != 'unknown']['tool_name'].value_counts().to_dict()
    }
}

# Add summary
filter_report['summary'] = {
    'input_artifacts': filter_report['input']['total_artifacts'],
    'output_common_artifacts': len(metadata),
    'output_ai_tools_artifacts': len(ai_metadata),
    'total_removed': filter_report['input']['total_artifacts'] - len(metadata),
    'removal_rate_pct': round((1 - len(metadata) / filter_report['input']['total_artifacts']) * 100, 1)
}

# Save filter report
with open(OUTPUT_FILTER_REPORT, 'w') as f:
    json.dump(filter_report, f, indent=2)
print(f"\nSaved: {OUTPUT_FILTER_REPORT}")

# Print summary
print("\n=== Filter Pipeline Summary ===")
print(f"Input:  {filter_report['summary']['input_artifacts']:,} artifacts")
print(f"Output (Common): {filter_report['summary']['output_common_artifacts']:,} artifacts")
print(f"Output (AI Tools): {filter_report['summary']['output_ai_tools_artifacts']:,} artifacts")
print(f"Total removed: {filter_report['summary']['total_removed']:,} ({filter_report['summary']['removal_rate_pct']}%)")

In [ ]:
print("\n" + "="*60)
print("FILTERING COMPLETE")
print("="*60)

print(f"\nOutput Directory: {DATA_DIR}")
print(f"\nOutput Files:")
print(f"  1. {OUTPUT_COMMON_EMBEDDINGS}")
print(f"  2. {OUTPUT_COMMON_METADATA}")
print(f"  3. {OUTPUT_AI_TOOLS_EMBEDDINGS}")
print(f"  4. {OUTPUT_AI_TOOLS_METADATA}")
print(f"  5. {OUTPUT_FILTER_REPORT}")

print(f"\n{'='*60}")
print("FINAL STATISTICS")
print(f"{'='*60}")

print(f"\nDataset 1 - Common Filters:")
print(f"  Artifacts: {len(metadata):,}")
print(f"  Repositories: {metadata['repo_name'].nunique():,}")

print(f"\nDataset 2 - AI Tools Only:")
print(f"  Artifacts: {len(ai_metadata):,}")
print(f"  Repositories: {ai_metadata['repo_name'].nunique():,}")

# Tool breakdown for AI Tools dataset
print(f"\n  AI Tools Breakdown:")
tool_counts = ai_metadata[ai_metadata['tool_name'] != 'unknown'].groupby('tool_name')['repo_name'].nunique().sort_values(ascending=False)
for tool, count in tool_counts.items():
    print(f"    - {tool}: {count} repos")

print(f"\n{'='*60}")